# 01 — Data Profiling & Cleaning
**Project:** Supply Chain Inventory Optimization (DataCo Global)

**Business problem:** DataCo is losing revenue to late deliveries and tying up cash in slow-moving stock.
Which products need tighter inventory control, what are the right reorder points, and what is the dollar impact?

**This notebook:** understand the raw data, document quality issues, and produce one clean order-line table
(`data/processed/order_lines_clean.csv`) that feeds SQL Server, the inventory model, and Power BI.

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', 60)

RAW = '../data/raw/DataCoSupplyChainDataset.csv'
df = pd.read_csv(RAW, encoding='ISO-8859-1')   # file contains non-UTF-8 characters
df.shape

(180519, 53)

## 1. Structure & missing values

In [2]:
df.dtypes.to_frame('dtype').T

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
dtype,str,int64,int64,float64,float64,str,int64,int64,str,str,str,str,str,int64,str,str,str,str,str,float64,int64,str,float64,float64,str,str,str,int64,str,int64,int64,float64,float64,int64,float64,float64,int64,float64,float64,float64,str,str,str,float64,int64,int64,float64,str,str,float64,int64,str,str


In [3]:
missing = df.isna().sum()
missing[missing > 0].to_frame('missing_rows').assign(pct=lambda x: (x.missing_rows / len(df) * 100).round(1))

,missing_rows,pct
Customer Lname,8,0.0
Customer Zipcode,3,0.0
Order Zipcode,155679,86.2
Product Description,180519,100.0


- `Product Description` is 100% empty and `Order Zipcode` is 86% empty, so both are dropped.
- Customer email, password, names and street are masked (`XXXXXXXXX`) or PII, and add nothing analytically, so they are dropped.

In [4]:
df[['Customer Email', 'Customer Password']].drop_duplicates().head()

,Customer Email,Customer Password
0,XXXXXXXXX,XXXXXXXXX


## 2. Redundant columns
Several columns are exact copies of each other. Verify before dropping:

In [5]:
pairs = [
    ('Order Customer Id', 'Customer Id'),
    ('Sales per customer', 'Order Item Total'),
    ('Benefit per order', 'Order Profit Per Order'),
    ('Order Item Cardprod Id', 'Product Card Id'),
    ('Product Category Id', 'Category Id'),
    ('Order Item Product Price', 'Product Price'),
]
pd.DataFrame([(a, b, np.allclose(df[a], df[b])) for a, b in pairs], columns=['col_a', 'col_b', 'identical'])

,col_a,col_b,identical
0,Order Customer Id,Customer Id,True
1,Sales per customer,Order Item Total,True
2,Benefit per order,Order Profit Per Order,True
3,Order Item Cardprod Id,Product Card Id,True
4,Product Category Id,Category Id,True
5,Order Item Product Price,Product Price,True


In [6]:
# Sanity check on the money columns: Sales = price x qty, Item Total = Sales - Discount
print('Sales vs price*qty max diff      :', (df['Sales'] - df['Order Item Product Price'] * df['Order Item Quantity']).abs().max().round(4))
print('Item Total vs Sales-Discount diff:', (df['Order Item Total'] - (df['Sales'] - df['Order Item Discount'])).abs().max().round(4))

Sales vs price*qty max diff      : 0.0
Item Total vs Sales-Discount diff: 0.01


## 3. Time coverage — a catalog break in Oct 2017

In [7]:
df['order_date'] = pd.to_datetime(df['order date (DateOrders)'])
monthly = df.groupby(df['order_date'].dt.to_period('M')).agg(order_lines=('Order Item Id', 'size'), products=('Product Name', 'nunique'), sales=('Sales', 'sum'))
monthly.tail(8)

,order_lines,products,sales
order_date,,,
2017-06,4951,42,1.032086e+06
2017-07,5318,42,1.104373e+06
2017-08,5305,42,1.109337e+06
2017-09,5189,53,1.143775e+06
2017-10,2255,20,1.073994e+06
2017-11,2055,8,6.269144e+05
2017-12,2124,14,5.039108e+05
2018-01,2123,10,3.316501e+05


In [8]:
pre  = set(df.loc[df.order_date <  '2017-10-01', 'Product Name'])
post = set(df.loc[df.order_date >= '2017-10-01', 'Product Name'])
print(f'Products before Oct-2017: {len(pre)} | after: {len(post)} | in both: {len(pre & post)}')

Products before Oct-2017: 100 | after: 31 | in both: 13


**Finding:** from October 2017 the product catalog changes almost completely (only 13 products continue) and volume halves.
Treating this as normal demand would distort every forecast and safety-stock number.

**Decision:** demand and inventory modelling uses the **stable window Jan 2015 – Sep 2017 (33 months)**. Later rows are kept, but flagged.

## 4. Delivery performance — first look

In [9]:
df['Delivery Status'].value_counts(normalize=True).mul(100).round(1)

Delivery Status
Late delivery        54.8
Advance shipping     23.0
Shipping on time     17.8
Shipping canceled     4.3
Name: proportion, dtype: float64

In [10]:
df.groupby('Shipping Mode').agg(
    order_lines=('Order Item Id', 'size'),
    late_rate=('Late_delivery_risk', 'mean'),
    promised_days=('Days for shipment (scheduled)', 'mean'),
    actual_days=('Days for shipping (real)', 'mean'),
).round(2).sort_values('late_rate', ascending=False)

,order_lines,late_rate,promised_days,actual_days
Shipping Mode,,,,
First Class,27814,0.95,1.0,2.00
Second Class,35216,0.77,2.0,3.99
Same Day,9737,0.46,0.0,0.48
Standard Class,107752,0.38,4.0,4.00


**Finding:** about 55% of order lines arrive late, but the problem is concentrated:
- **First Class** promises 1 day and *always* takes 2, so it is ~95% late.
- **Second Class** promises 2 days and averages ~4.
- **Standard Class** mostly delivers on its 4-day promise.

That looks like a **mis-set delivery promise (SLA)** rather than random warehouse failure. This gets investigated properly in a later notebook.

In [11]:
df.groupby('Order Status')['Late_delivery_risk'].mean().round(2).sort_values()

Order Status
CANCELED           0.00
SUSPECTED_FRAUD    0.00
ON_HOLD            0.56
CLOSED             0.57
PAYMENT_REVIEW     0.57
PROCESSING         0.57
COMPLETE           0.57
PENDING            0.58
PENDING_PAYMENT    0.58
Name: Late_delivery_risk, dtype: float64

CANCELED and SUSPECTED_FRAUD orders never ship (late risk = 0). They are flagged and excluded from delivery and demand analysis.

## 5. Build the clean table

In [12]:
drop_cols = [
    # PII / masked
    'Customer Email', 'Customer Password', 'Customer Fname', 'Customer Lname', 'Customer Street', 'Customer Zipcode',
    # empty / constant / unused
    'Product Description', 'Order Zipcode', 'Product Status', 'Product Image', 'Latitude', 'Longitude',
    # exact duplicates (verified above)
    'Order Customer Id', 'Sales per customer', 'Benefit per order', 'Order Item Cardprod Id',
    'Product Category Id', 'Product Price',
    # replaced by parsed dates
    'order date (DateOrders)', 'shipping date (DateOrders)',
]
clean = df.copy()
clean['ship_date'] = pd.to_datetime(clean['shipping date (DateOrders)'])
clean = clean.drop(columns=drop_cols)

clean.columns = (clean.columns.str.strip().str.lower()
                 .str.replace(r'[^a-z0-9]+', '_', regex=True).str.strip('_'))
clean = clean.rename(columns={
    'days_for_shipping_real': 'ship_days_actual',
    'days_for_shipment_scheduled': 'ship_days_promised',
    'type': 'payment_type',
    'product_card_id': 'product_id',
    'order_item_product_price': 'unit_price',
    'order_profit_per_order': 'order_item_profit',
})

# Derived fields
clean['delay_days']     = clean['ship_days_actual'] - clean['ship_days_promised']
clean['is_cancelled']   = clean['order_status'].isin(['CANCELED', 'SUSPECTED_FRAUD']).astype(int)
clean['in_model_window'] = (clean['order_date'] < '2017-10-01').astype(int)
clean['order_month']    = clean['order_date'].dt.to_period('M').dt.to_timestamp()

# Raw money fields carry float noise (e.g. 9.989999771); round so SQL DECIMAL loads cleanly
num_cols = clean.select_dtypes('float').columns
clean[num_cols] = clean[num_cols].round(4)

# Some text values carry stray whitespace (e.g. 'Smart watch ')
txt_cols = clean.select_dtypes(include=['object', 'string']).columns
clean[txt_cols] = clean[txt_cols].apply(lambda s: s.str.strip())

clean.shape

(180519, 39)

In [13]:
clean.head(3).T

,0,1,2
payment_type,DEBIT,TRANSFER,CASH
ship_days_actual,3,5,4
ship_days_promised,4,4,4
delivery_status,Advance shipping,Late delivery,Shipping on time
late_delivery_risk,0,1,0
category_id,73,73,73
category_name,Sporting Goods,Sporting Goods,Sporting Goods
customer_city,Caguas,Caguas,San Jose
customer_country,Puerto Rico,Puerto Rico,EE. UU.
customer_id,20755,19492,19491


In [14]:
# Final checks
assert clean['order_item_id'].is_unique
assert clean.isna().sum().sum() < 10, clean.isna().sum()[lambda s: s > 0]
print(clean.isna().sum()[lambda s: s > 0])
print('Rows in model window, not cancelled:', int(((clean.in_model_window == 1) & (clean.is_cancelled == 0)).sum()))

Series([], dtype: int64)
Rows in model window, not cancelled: 164572


In [15]:
import os
os.makedirs('../data/processed', exist_ok=True)
clean.to_csv('../data/processed/order_lines_clean.csv', index=False)
print('Saved:', clean.shape)

Saved: (180519, 39)


## Summary
| Item | Result |
|---|---|
| Raw rows / columns | 180,519 / 53 |
| Clean columns | see shape above (PII, empty and duplicate columns removed) |
| Products / categories | 118 / 50 |
| Date range | Jan 2015 – Jan 2018 |
| Modelling window | Jan 2015 – Sep 2017 (catalog break in Oct 2017) |
| Late delivery rate | ~55%, driven by First and Second Class SLAs |

**Next:** load the clean table into SQL Server, then run ABC-XYZ demand segmentation.